# LeetCode 3940 — Limit Occurrences in Sorted Array bằng Python

Link đề: https://leetcode.com/problems/limit-occurrences-in-sorted-array/ · Độ khó: **Easy** · Chủ đề: Array, Two Pointers, Sorted Array

Skeleton LeetCode:

```python
class Solution:
    def limitOccurrences(self, nums: List[int], k: int) -> List[int]:
        ...
```

> Ghi chú: tên bài, ví dụ, constraints và chữ ký hàm được đối chiếu qua leetcode.doocs.org (Easy, không Premium); phần phát biểu đề được viết lại bằng lời của mình. Slug URL được suy ra từ tên bài.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mảng `nums` đã **sắp xếp không giảm**; giữ lại **tối đa `k` lần** mỗi giá trị, giữ nguyên thứ tự.
2. Nắm kỹ thuật **con trỏ ghi (write pointer)**: phần tử `nums[i]` được giữ khi nó khác phần tử ở vị trí `w - k` của kết quả.
3. Biết dùng `collections.Counter`, `itertools.groupby` và lát cắt `nums[:w]`.
4. Viết 4 cách (đếm, `groupby`, danh sách kết quả, ghi tại chỗ O(1) bộ nhớ phụ) và đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh lỗi: khử trùng lặp hoàn toàn (`set`), xóa phần tử khi đang duyệt, chỉ so với phần tử liền trước (chỉ đúng khi `k = 1`).

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` **sắp xếp không giảm** và số nguyên `k`. Hãy trả về một mảng trong đó **mỗi giá trị xuất hiện tối đa `k` lần**, giữ nguyên thứ tự tương đối. Nếu một giá trị xuất hiện ít nhất `k` lần trong `nums` thì nó xuất hiện **đúng `k` lần** trong kết quả; nếu ít hơn `k` thì giữ nguyên số lần.

| Ví dụ | `nums`, `k` | Output | Giải thích |
|---|---|---|---|
| 1 | `[1,1,1,2,2,3]`, `k=2` | `[1,1,2,2,3]` | `1` xuất hiện 3 lần -> giữ 2; `2` đủ 2 lần; `3` một lần |
| 2 | `[1,2,3]`, `k=1` | `[1,2,3]` | Các giá trị đều khác nhau nên không đổi |

Constraints: `1 <= len(nums) <= 100`; `1 <= nums[i] <= 100`; `nums` sắp xếp không giảm; `1 <= k <= len(nums)`.

Câu hỏi nâng cao của đề (follow-up): làm **tại chỗ** với `O(1)` bộ nhớ phụ (không tính mảng trả về).

Khi `k = 1` bài chính là "Remove Duplicates from Sorted Array" (26); khi `k = 2` là bài 80.

In [1]:
examples = [([1, 1, 1, 2, 2, 3], 2, [1, 1, 2, 2, 3]), ([1, 2, 3], 1, [1, 2, 3])]
for nums, k, expected in examples:
    print(f"nums={nums}, k={k} -> expected={expected}")
    for v in sorted(set(nums)):
        print(f"   giá trị {v}: xuất hiện {nums.count(v)} lần -> giữ {min(nums.count(v), k)}")

nums=[1, 1, 1, 2, 2, 3], k=2 -> expected=[1, 1, 2, 2, 3]
   giá trị 1: xuất hiện 3 lần -> giữ 2
   giá trị 2: xuất hiện 2 lần -> giữ 2
   giá trị 3: xuất hiện 1 lần -> giữ 1
nums=[1, 2, 3], k=1 -> expected=[1, 2, 3]
   giá trị 1: xuất hiện 1 lần -> giữ 1
   giá trị 2: xuất hiện 1 lần -> giữ 1
   giá trị 3: xuất hiện 1 lần -> giữ 1


## 3. Kiến thức nền

### 3.1. Mảng đã sắp xếp: các giá trị bằng nhau đứng **liền nhau**

Trong mảng sắp xếp không giảm, mọi bản sao của một giá trị tạo thành **một khối liên tiếp** (run). Điều này cho phép đếm mà không cần từ điển, chỉ cần so sánh phần tử với hàng xóm.

In [2]:
nums = [1, 1, 1, 2, 2, 3]
print([nums[i] == nums[i - 1] for i in range(1, len(nums))])

[True, True, False, True, False]


### 3.2. `collections.Counter`

`Counter(nums)` cho từ điển `giá_trị -> số lần`. Duyệt `Counter` theo thứ tự chèn (Python 3.7+), tức thứ tự xuất hiện đầu tiên. Vì `nums` sắp xếp nên cũng là thứ tự tăng dần.

In [3]:
from collections import Counter
c = Counter([1, 1, 1, 2, 2, 3])
print(c, list(c.items()))

Counter({1: 3, 2: 2, 3: 1}) [(1, 3), (2, 2), (3, 1)]


### 3.3. `itertools.groupby`

`groupby(nums)` gom các phần tử **bằng nhau liền kề** thành nhóm `(giá_trị, iterator)`. Với mảng sắp xếp, mỗi giá trị đúng một nhóm. Muốn biết độ dài nhóm, đổi iterator sang list: `len(list(g))`.

In [4]:
from itertools import groupby
for v, g in groupby([1, 1, 1, 2, 2, 3]):
    grp = list(g)
    print(v, grp, len(grp))

1 [1, 1, 1] 3
2 [2, 2] 2
3 [3] 1


### 3.4. Nhân list: `[v] * m`

`[7] * 3` là `[7, 7, 7]` (cẩn thận với list lồng nhau, nhưng ở đây phần tử là số nên an toàn). `[v] * min(cnt, k)` cho đúng số bản sao cần giữ.

In [5]:
print([7] * 3, [7] * 0, [5] * min(4, 2))

[7, 7, 7] [] [5, 5]


### 3.5. Con trỏ ghi (write pointer) và lát cắt `nums[:w]`

Ta duyệt bằng con trỏ đọc `i`; con trỏ ghi `w` là vị trí để ghi phần tử được giữ. Sau khi duyệt xong, `nums[:w]` là kết quả. Đây là kỹ thuật **tại chỗ** kinh điển, không cần mảng phụ.

In [6]:
a = [1, 1, 1, 2, 2, 3]
w = 0
for x in a:
    if w < 2 or a[w - 2] != x:
        a[w] = x
        w += 1
print(w, a[:w], a)

5 [1, 1, 2, 2, 3] [1, 1, 2, 2, 3, 3]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Vì giá trị bằng nhau đứng liền nhau, mỗi khối `v v v ... v` (độ dài `c`) cần bị **cắt còn `min(c, k)`** phần tử. Hãy "đứng ở cuối kết quả đang dựng" và hỏi mỗi phần tử mới `x`: **trong kết quả hiện tại đã có `k` bản sao của `x` chưa?**

Vì kết quả cũng sắp xếp, `k` bản sao gần nhất của `x` (nếu có) đứng ngay cuối. Do đó chỉ cần nhìn phần tử **cách cuối `k` vị trí**: nếu `kq[-k] == x` thì đã đủ `k` bản sao (cả `k` phần tử cuối đều bằng `x`) -> bỏ `x`; ngược lại giữ.

Tính tay `[1,1,1,2,2,3]`, `k = 2`:

| phần tử | kq trước đó | `kq[-2]` | quyết định |
|---|---|---|---|
| 1 | `[]` | (không có) | giữ -> `[1]` |
| 1 | `[1]` | (không có) | giữ -> `[1,1]` |
| 1 | `[1,1]` | `1 == 1` | bỏ |
| 2 | `[1,1]` | `1 != 2` | giữ -> `[1,1,2]` |
| 2 | `[1,1,2]` | `1 != 2` | giữ -> `[1,1,2,2]` |
| 3 | `[1,1,2,2]` | `2 != 3` | giữ -> `[1,1,2,2,3]` |

In [7]:
nums, k = [1, 1, 1, 2, 2, 3], 2
res = []
for x in nums:
    keep = len(res) < k or res[-k] != x
    print(f"x={x}, kq={res}, giữ={keep}")
    if keep:
        res.append(x)
print(res)

x=1, kq=[], giữ=True
x=1, kq=[1], giữ=True
x=1, kq=[1, 1], giữ=False
x=2, kq=[1, 1], giữ=True
x=2, kq=[1, 1, 2], giữ=True
x=3, kq=[1, 1, 2, 2], giữ=True
[1, 1, 2, 2, 3]


## 5. Cách 1 — Brute Force / baseline: đếm bằng `Counter`

Đếm số lần mỗi giá trị, rồi dựng kết quả: với mỗi giá trị theo thứ tự tăng, thêm `min(đếm, k)` bản sao.

Cách này đúng cho **mọi** mảng (sắp xếp hay không, nếu duyệt giá trị theo thứ tự xuất hiện lần đầu) nhưng dùng thêm từ điển, và chưa tận dụng tính sắp xếp. Đây là "nút thắt": dùng `O(số giá trị khác nhau)` bộ nhớ thừa.

In [8]:
from collections import Counter

class SolutionCounter:
    def limitOccurrences(self, nums, k):
        cnt = Counter(nums)
        res = []
        for v, c in cnt.items():
            res.extend([v] * min(c, k))
        return res

print(SolutionCounter().limitOccurrences([1, 1, 1, 2, 2, 3], 2), SolutionCounter().limitOccurrences([1, 2, 3], 1))

[1, 1, 2, 2, 3] [1, 2, 3]


**Độ phức tạp:** Time `O(n)`, Space `O(d)` cho từ điển (`d` = số giá trị khác nhau) cộng mảng kết quả.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** vì `nums` đã sắp xếp nên **"đã có `k` bản sao của `x` trong kết quả"** tương đương **`kq[-k] == x`** (hoặc, tại chỗ, `nums[w - k] == x`). Không cần đếm, không cần từ điển.

**Quy tắc "nếu... thì...":**

- nếu `len(kq) < k` thì chắc chắn giữ `x` (chưa thể đủ `k` bản sao);
- nếu `kq[-k] == x` thì bỏ `x` (đã có `k` bản sao: `kq[-k] .. kq[-1]` đều bằng `x` do sắp xếp);
- ngược lại giữ `x`.

**Invariant:** sau mỗi bước, `kq` là mảng sắp xếp, mỗi giá trị xuất hiện `<= k` lần, và mỗi giá trị đã xét có đúng `min(đếm_đã_thấy, k)` bản sao trong `kq`.

Với `k = 1` đây là bài "bỏ trùng"; với `k = 2` là bài 80.

## 7. Các cách tiếp theo

### Cách 2 — `groupby` theo khối

Mỗi khối liền kề cùng giá trị: giữ `min(độ dài, k)` phần tử. Không cần từ điển, tận dụng đúng "khối liên tiếp".

In [9]:
from itertools import groupby

class SolutionGroupBy:
    def limitOccurrences(self, nums, k):
        res = []
        for v, g in groupby(nums):
            c = sum(1 for _ in g)
            res.extend([v] * min(c, k))
        return res

print(SolutionGroupBy().limitOccurrences([1, 1, 1, 2, 2, 3], 2), SolutionGroupBy().limitOccurrences([1, 2, 3], 1))

[1, 1, 2, 2, 3] [1, 2, 3]


**Độ phức tạp:** Time `O(n)`, Space `O(1)` phụ (không tính kết quả).

### Cách 3 — Dựng kết quả với phép kiểm tra `res[-k]`

Đây là cách "dựng list mới" của quy tắc ở mục 6, dễ chứng minh và không sửa đầu vào.

In [10]:
class SolutionBuild:
    def limitOccurrences(self, nums, k):
        res = []
        for x in nums:
            if len(res) < k or res[-k] != x:
                res.append(x)
        return res

print(SolutionBuild().limitOccurrences([1, 1, 1, 2, 2, 3], 2), SolutionBuild().limitOccurrences([1, 2, 3], 1))

[1, 1, 2, 2, 3] [1, 2, 3]


**Độ phức tạp:** Time `O(n)`, Space `O(n)` cho kết quả (không tính, bộ nhớ phụ `O(1)`).

### Cách 4 — Ghi tại chỗ (follow-up): con trỏ ghi `w`

Không tạo list mới: ghi thẳng vào `nums` ở vị trí `w`, điều kiện giữ là `w < k or nums[w - k] != x`. Cuối cùng trả `nums[:w]`. Chú ý: hàm này **sửa `nums`** (đúng tinh thần follow-up), nên khi test phải truyền bản sao.

In [11]:
class SolutionInPlace:
    def limitOccurrences(self, nums, k):
        w = 0
        for x in nums:
            if w < k or nums[w - k] != x:
                nums[w] = x
                w += 1
        return nums[:w]

print(SolutionInPlace().limitOccurrences([1, 1, 1, 2, 2, 3], 2), SolutionInPlace().limitOccurrences([1, 2, 3], 1))

[1, 1, 2, 2, 3] [1, 2, 3]


**Độ phức tạp:** Time `O(n)`, Space `O(1)` bộ nhớ phụ (lát cắt trả về không tính theo đề).

Chú ý vì sao `for x in nums` vẫn an toàn dù ta ghi vào `nums`: ta chỉ ghi ở `nums[w]` với `w <= i` (con trỏ ghi không bao giờ vượt con trỏ đọc), nên chưa đọc tới ô đã bị ghi đè.

## 8. TRACE từng bước

Trace Cách 4 trên `nums = [1,1,1,2,2,3]`, `k = 2`:

| i | `x` | `w` trước | `nums[w-k]` | giữ? | `nums` sau |
|---|---|---|---|---|---|
| 0 | 1 | 0 | (w<k) | có | `[1,1,1,2,2,3]`, `w=1` |
| 1 | 1 | 1 | (w<k) | có | `w=2` |
| 2 | 1 | 2 | `nums[0]=1` | **không** | `w=2` |
| 3 | 2 | 2 | `nums[0]=1` | có | `[1,1,2,2,2,3]`, `w=3` |
| 4 | 2 | 3 | `nums[1]=1` | có | `[1,1,2,2,2,3]`, `w=4` |
| 5 | 3 | 4 | `nums[2]=2` | có | `[1,1,2,2,3,3]`, `w=5` |

Kết quả `nums[:5] = [1,1,2,2,3]`.

In [12]:
def trace_in_place(nums, k):
    a = list(nums)
    w = 0
    print("bắt đầu:", a, "k =", k)
    for i, x in enumerate(nums):
        if w < k:
            keep, why = True, "w < k"
        else:
            keep, why = a[w - k] != x, f"a[{w - k}]={a[w - k]} {'!=' if a[w - k] != x else '=='} {x}"
        if keep:
            a[w] = x
            w += 1
        print(f"  i={i}, x={x}, {why} -> {'giữ ' if keep else 'bỏ  '} a={a}, w={w}")
    print("kết quả:", a[:w])
    return a[:w]

assert trace_in_place([1, 1, 1, 2, 2, 3], 2) == [1, 1, 2, 2, 3]
assert trace_in_place([1, 2, 3], 1) == [1, 2, 3]

bắt đầu: [1, 1, 1, 2, 2, 3] k = 2
  i=0, x=1, w < k -> giữ  a=[1, 1, 1, 2, 2, 3], w=1
  i=1, x=1, w < k -> giữ  a=[1, 1, 1, 2, 2, 3], w=2
  i=2, x=1, a[0]=1 == 1 -> bỏ   a=[1, 1, 1, 2, 2, 3], w=2
  i=3, x=2, a[0]=1 != 2 -> giữ  a=[1, 1, 2, 2, 2, 3], w=3
  i=4, x=2, a[1]=1 != 2 -> giữ  a=[1, 1, 2, 2, 2, 3], w=4
  i=5, x=3, a[2]=2 != 3 -> giữ  a=[1, 1, 2, 2, 3, 3], w=5
kết quả: [1, 1, 2, 2, 3]
bắt đầu: [1, 2, 3] k = 1
  i=0, x=1, w < k -> giữ  a=[1, 2, 3], w=1
  i=1, x=2, a[0]=1 != 2 -> giữ  a=[1, 2, 3], w=2
  i=2, x=3, a[1]=2 != 3 -> giữ  a=[1, 2, 3], w=3
kết quả: [1, 2, 3]


In [13]:
def trace_groupby(nums, k):
    from itertools import groupby
    res = []
    for v, g in groupby(nums):
        c = len(list(g))
        keep = min(c, k)
        print(f"khối {v}: dài {c} -> giữ {keep}")
        res += [v] * keep
    print("kết quả:", res)
    return res

assert trace_groupby([1, 1, 1, 2, 2, 3], 2) == [1, 1, 2, 2, 3]

khối 1: dài 3 -> giữ 2
khối 2: dài 2 -> giữ 2
khối 3: dài 1 -> giữ 1
kết quả: [1, 1, 2, 2, 3]


## 9. Code nộp LeetCode

Dùng Cách 3 (dựng list mới, không sửa đầu vào). Nếu đề yêu cầu tại chỗ, đổi sang Cách 4.

In [14]:
from typing import List, Optional


class Solution:
    def limitOccurrences(self, nums: List[int], k: int) -> List[int]:
        res = []
        for x in nums:
            if len(res) < k or res[-k] != x:
                res.append(x)
        return res


print(Solution().limitOccurrences([1, 1, 1, 2, 2, 3], 2), Solution().limitOccurrences([1, 2, 3], 1))

[1, 1, 2, 2, 3] [1, 2, 3]


## 10. Vì sao thuật toán đúng

Xét thuật toán dựng `res`. Ta chứng minh bằng quy nạp theo số phần tử đã xử lý rằng: *với mỗi giá trị `v` trong các phần tử đã xét, `res` chứa đúng `min(c_v, k)` bản sao của `v`*, với `c_v` là số lần `v` đã thấy, và `res` luôn sắp xếp.

1. **Cơ sở.** Trước khi xử lý gì, `res = []`, hiển nhiên đúng.
2. **Bước quy nạp.** Xét `x` tiếp theo. Vì `nums` sắp xếp nên `x >= mọi phần tử của res`, và các bản sao của `x` trong `res` (nếu có) là **hậu tố liên tiếp** của `res`.
3. Nếu `len(res) < k` thì `res` chưa thể có `k` bản sao của `x` -> giữ là đúng.
4. Nếu `len(res) >= k`: `res[-k] == x` nghĩa là `k` phần tử cuối đều bằng `x` (do sắp xếp: `res[-k] <= ... <= res[-1] <= x`, mà hai đầu cùng bằng `x`). Khi đó đã đủ `k` bản sao, bỏ `x`. Nếu `res[-k] != x` thì có ít hơn `k` bản sao của `x` -> giữ.
5. Vì thế số bản sao của mỗi giá trị trong `res` đúng bằng `min(c_v, k)` và thứ tự tương đối được giữ (ta chỉ **xóa** bớt, không đổi chỗ). Các cách khác (Counter, groupby) tính cùng đại lượng `min(c_v, k)` theo định nghĩa trực tiếp.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)` (`<= 100`), `d` = số giá trị khác nhau.

- Cách 1 (Counter): Time `O(n)`, Space `O(d)` + kết quả.
- Cách 2 (`groupby`): Time `O(n)`, Space `O(1)` phụ + kết quả.
- Cách 3 (nộp): Time `O(n)`, Space kết quả `O(n)` (phụ `O(1)`).
- Cách 4 (tại chỗ): Time `O(n)`, Space phụ `O(1)`; lát cắt `nums[:w]` là kết quả trả về.

`O(n)` là tối ưu vì phải đọc mọi phần tử. Mảng sắp xếp còn cho phép dùng tìm nhị phân để nhảy cả khối (khi khối rất dài), nhưng với `n <= 100` không cần.

## 12. Bộ test

### Cách kiểm thử trong notebook này

Chúng ta dùng ba lớp kiểm thử, từ dễ đến khó:

1. **Test cố định**: ví dụ của đề + các trường hợp biên tự nghĩ ra. Mỗi test là một `assert hàm(input) == kết_quả_mong_đợi`. Nếu sai, Python ném `AssertionError` và dừng ngay, kèm thông tin ta in ra để dễ lần lại.
2. **Test ngẫu nhiên có `random.seed`**: gọi `random.seed(số)` trước vòng lặp để **mỗi lần chạy lại sinh đúng cùng dữ liệu** (tái lập được). Input luôn được sinh **trong đúng constraints** của đề.
3. **Đối chiếu nhiều cách (oracle)**: một cách rất đơn giản, hiển nhiên đúng (brute force) đóng vai "trọng tài"; mọi cách nhanh hơn phải cho cùng kết quả với trọng tài trên mọi input ngẫu nhiên.

In [15]:
import random

def ref(nums, k):
    # trọng tài: định nghĩa trực tiếp bằng đếm
    out, cnt = [], {}
    for x in nums:
        cnt[x] = cnt.get(x, 0) + 1
        if cnt[x] <= k:
            out.append(x)
    return out

solutions = [SolutionCounter(), SolutionGroupBy(), SolutionBuild(), SolutionInPlace(), Solution()]

fixed = [
    ([1, 1, 1, 2, 2, 3], 2, [1, 1, 2, 2, 3]), ([1, 2, 3], 1, [1, 2, 3]),
    ([1], 1, [1]), ([5, 5], 1, [5]), ([5, 5], 2, [5, 5]), ([2, 2, 2, 2], 3, [2, 2, 2]),
    ([1, 1, 2, 2, 3, 3], 1, [1, 2, 3]), ([1, 1, 2, 2, 3, 3], 6, [1, 1, 2, 2, 3, 3]),
    ([1, 1, 1, 1, 2], 2, [1, 1, 2]), ([1, 2, 2, 2, 3, 3, 3, 3], 2, [1, 2, 2, 3, 3]),
]
for nums, k, expected in fixed:
    assert ref(nums, k) == expected
    for sol in solutions:
        arg = list(nums)
        assert sol.limitOccurrences(arg, k) == expected, (type(sol).__name__, nums, k)
print("test cố định OK:", len(fixed), "ca")

random.seed(3940)
for _ in range(500):
    n = random.randint(1, 100)
    nums = sorted(random.randint(1, random.choice([3, 10, 100])) for _ in range(n))
    k = random.randint(1, n)
    expected = ref(nums, k)
    for sol in solutions:
        assert sol.limitOccurrences(list(nums), k) == expected, (type(sol).__name__, nums, k)
print("test ngẫu nhiên OK")

test cố định OK: 10 ca
test ngẫu nhiên OK


### Test thuộc tính (property test)

(1) Mỗi giá trị xuất hiện `<= k` lần; số lần = `min(đếm gốc, k)`. (2) Kết quả vẫn sắp xếp và là **dãy con** của `nums`. (3) Đầu vào **không bị sửa** (với cách nộp). (4) `k >= n` thì kết quả bằng `nums`.

In [16]:
from collections import Counter

random.seed(39400)
for _ in range(400):
    n = random.randint(1, 60)
    nums = sorted(random.randint(1, 8) for _ in range(n))
    k = random.randint(1, n)
    snapshot = list(nums)
    out = Solution().limitOccurrences(nums, k)
    assert nums == snapshot
    c0, c1 = Counter(nums), Counter(out)
    assert all(c1[v] == min(c0[v], k) for v in c0) and set(c1) == set(c0)
    assert out == sorted(out)
    it = iter(nums)
    assert all(any(y == x for y in it) for x in out)      # out là dãy con của nums
    assert Solution().limitOccurrences(nums, n) == nums
print("property test OK")

property test OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: dùng `set` / `sorted(set(...))` (bỏ trùng hoàn toàn)

`set` loại **mọi** bản sao, chỉ giữ 1; đề cho giữ tới `k` bản sao.

In [17]:
def wrong_set(nums, k):
    return sorted(set(nums))                          # SAI: chỉ đúng khi k = 1

print(wrong_set([1, 1, 1, 2, 2, 3], 2), "!=", Solution().limitOccurrences([1, 1, 1, 2, 2, 3], 2))
assert wrong_set([1, 1, 1, 2, 2, 3], 2) == [1, 2, 3] and Solution().limitOccurrences([1, 1, 1, 2, 2, 3], 2) == [1, 1, 2, 2, 3]
assert wrong_set([1, 1, 2], 1) == Solution().limitOccurrences([1, 1, 2], 1)

[1, 2, 3] != [1, 1, 2, 2, 3]


### Lỗi 2: chỉ so với phần tử **liền trước** (`nums[i] != nums[i-1]`)

Đó là điều kiện của bài `k = 1`. Với `k = 2` cần so với phần tử cách `k` vị trí trong kết quả.

In [18]:
def wrong_prev_only(nums, k):
    res = []
    for x in nums:
        if not res or res[-1] != x:                   # SAI: phải res[-k]
            res.append(x)
    return res

print(wrong_prev_only([1, 1, 1, 2, 2, 3], 2), "!=", Solution().limitOccurrences([1, 1, 1, 2, 2, 3], 2))
assert wrong_prev_only([1, 1, 1, 2, 2, 3], 2) == [1, 2, 3]

[1, 2, 3] != [1, 1, 2, 2, 3]


### Lỗi 3: xóa phần tử trong khi đang duyệt chính list đó

`for x in nums: nums.remove(x)` làm các chỉ số dịch chuyển, vòng lặp **bỏ qua** phần tử. Kết quả sai (và khó đoán).

In [19]:
def wrong_remove_while_iterating(nums, k):
    a = list(nums)
    seen = {}
    for x in a:                                       # SAI: sửa a khi đang duyệt a
        seen[x] = seen.get(x, 0) + 1
        if seen[x] > k:
            a.remove(x)
    return a

bad = wrong_remove_while_iterating([1, 1, 1, 1, 1, 2], 1)
print(bad, "!=", Solution().limitOccurrences([1, 1, 1, 1, 1, 2], 1))
assert bad != [1, 2] and Solution().limitOccurrences([1, 1, 1, 1, 1, 2], 1) == [1, 2]

[1, 1, 1, 2] != [1, 2]


### Lỗi 4: quên điều kiện `len(res) < k` (truy cập `res[-k]` khi `res` quá ngắn)

Khi `res` có ít hơn `k` phần tử, `res[-k]` gây `IndexError` (hoặc trả sai ý nếu dùng chỉ số khác).

In [20]:
def wrong_no_guard(nums, k):
    res = []
    for x in nums:
        if res[-k] != x:                              # SAI: lúc res còn ngắn
            res.append(x)
    return res

try:
    wrong_no_guard([1, 1, 2], 2)
except IndexError as e:
    print("IndexError:", e)
assert Solution().limitOccurrences([1, 1, 2], 2) == [1, 1, 2]

IndexError: list index out of range


### Lỗi 5: bỏ nhầm `<` thành `<=` trong số bản sao

Giữ `k` bản sao nghĩa là `nums[w - k] != x` (không phải `nums[w - k - 1]`): quá nhiều bản sao lọt qua. Test với `[1,1,1,1]`, `k = 2`.

In [21]:
def wrong_off_by_one(nums, k):
    res = []
    for x in nums:
        if len(res) <= k or res[-k - 1] != x:         # SAI: lệch một -> giữ k + 1 bản sao
            res.append(x)
    return res

print(wrong_off_by_one([1, 1, 1, 1], 2), "!=", Solution().limitOccurrences([1, 1, 1, 1], 2))
assert wrong_off_by_one([1, 1, 1, 1], 2) == [1, 1, 1] and Solution().limitOccurrences([1, 1, 1, 1], 2) == [1, 1]

[1, 1, 1] != [1, 1]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Mảng **sắp xếp**, mỗi giá trị tối đa `k` lần" -> **con trỏ ghi + so với `nums[w - k]`** (bài 26 là `k = 1`, bài 80 là `k = 2`).
- Hai dạng chiều: cách "dựng list mới" (`res[-k]`) và "ghi tại chỗ" (`nums[w - k]`) hoàn toàn tương đương; chọn dạng đầu để dễ gỡ lỗi.
- Khi mảng **không** sắp xếp, mẹo `res[-k]` mất hiệu lực: phải dùng `Counter`/từ điển.
- `groupby` rất tiện cho mọi bài "xử lý theo từng khối liền kề bằng nhau".
- Nhớ `len(res) < k or ...` để tránh chỉ số âm vượt biên; luôn thử `k = 1`, `k = len(nums)`.
- Đọc constraints: `n <= 100` nên mọi cách `O(n)` hay `O(n log n)` đều đủ.

## 15. Biến thể của bài

1. **Remove Duplicates from Sorted Array** (26): `k = 1`, trả về độ dài.
2. **Remove Duplicates from Sorted Array II** (80): `k = 2`, tại chỗ.
3. **Remove Element** (27): xóa một giá trị cho trước.
4. **Move Zeroes** (283): con trỏ ghi cho các phần tử khác 0.
5. Biến thể: giới hạn **tổng số bản sao** theo từng giá trị khác nhau (dict `limit[v]`).
6. Biến thể: mảng **không sắp xếp** (cần `Counter`), hoặc yêu cầu **giữ lại `k` lần xuất hiện cuối cùng** thay vì `k` lần đầu.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. `Counter` | đếm rồi nhân bản `min(c, k)` | `O(n)` | `O(d)` | Mảng bất kỳ (không cần sắp xếp) |
| 2. `groupby` | xử lý từng khối liền kề | `O(n)` | `O(1)` phụ | Mảng sắp xếp, code rõ ràng |
| 3. Dựng list với `res[-k]` | so với phần tử cách `k` | `O(n)` | `O(n)` kết quả | **Nộp bài**, không sửa đầu vào |
| 4. Ghi tại chỗ | con trỏ ghi `w`, `nums[w-k]` | `O(n)` | `O(1)` phụ | Follow-up của đề |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().limitOccurrences([4, 4, 4, 4], 3)` và `Solution().limitOccurrences([1, 2, 2, 3, 3, 3], 1)` ra gì?

**Bài 2 (tính tay):** với `nums = [1,1,2,2,2,3,3,3,3]`, `k = 2`: ghi lại `res` sau mỗi phần tử.

**Bài 3 (điền TODO):** hoàn thành `limit_last_k(nums, k)`: giữ `k` lần xuất hiện **cuối cùng** của mỗi giá trị (mảng sắp xếp).

**Bài 4 (tự viết lại):** viết `limit_in_place_count(nums, k)` sửa `nums` tại chỗ và **trả về độ dài** `w` (kiểu LeetCode 80).

**Bài 5 (mini challenge):** viết `limit_unsorted(nums, k)` cho mảng **không** sắp xếp: giữ `k` lần xuất hiện **đầu tiên** của mỗi giá trị, giữ thứ tự gốc.

In [22]:
# Bài 1: đoán rồi chạy
print(Solution().limitOccurrences([4, 4, 4, 4], 3), Solution().limitOccurrences([1, 2, 2, 3, 3, 3], 1))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def limit_last_k_todo(nums, k):
    # TODO: duyệt ngược (reversed), áp dụng cùng quy tắc, rồi đảo lại kết quả
    return nums

[4, 4, 4] [1, 2, 3]


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: `[4,4,4,4]` giữ 3 bản sao. Bài 2: ghi `res` từng bước theo bảng ở mục 4.

**Mức 2 (vừa):** Bài 3: duyệt `reversed(nums)` với quy tắc `res[-k]`, rồi đảo kết quả. Bài 4: trả về `w` thay vì lát cắt. Bài 5: dùng `dict` đếm `cnt[x]` và chỉ thêm khi `cnt[x] <= k`.

**Mức 3 (gần đáp án):** Bài 1: `[4,4,4]` và `[1,2,3]`. Bài 2: `res = [1,1,2,2,3,3]`. Bài 3: `[1,1,1,2]`, `k=2` -> `[1,1,2]` (giữ 2 lần cuối của `1`). Bài 5: `[3,1,3,1,3]`, `k=2` -> `[3,1,3,1]`.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1, 2
assert Solution().limitOccurrences([4, 4, 4, 4], 3) == [4, 4, 4]
assert Solution().limitOccurrences([1, 2, 2, 3, 3, 3], 1) == [1, 2, 3]
assert Solution().limitOccurrences([1, 1, 2, 2, 2, 3, 3, 3, 3], 2) == [1, 1, 2, 2, 3, 3]

# Bài 3
def limit_last_k(nums, k):
    res = []
    for x in reversed(nums):
        if len(res) < k or res[-k] != x:
            res.append(x)
    return res[::-1]

assert limit_last_k([1, 1, 1, 2], 2) == [1, 1, 2] and limit_last_k([1, 2, 3], 1) == [1, 2, 3]
assert limit_last_k([1, 1, 1, 1], 2) == [1, 1]

# Bài 4
def limit_in_place_count(nums, k):
    w = 0
    for x in nums:
        if w < k or nums[w - k] != x:
            nums[w] = x
            w += 1
    return w

arr = [1, 1, 1, 2, 2, 3]
w = limit_in_place_count(arr, 2)
assert w == 5 and arr[:w] == [1, 1, 2, 2, 3]

# Bài 5
def limit_unsorted(nums, k):
    cnt, out = {}, []
    for x in nums:
        cnt[x] = cnt.get(x, 0) + 1
        if cnt[x] <= k:
            out.append(x)
    return out

assert limit_unsorted([3, 1, 3, 1, 3], 2) == [3, 1, 3, 1] and limit_unsorted([2, 2, 2], 1) == [2]
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

### Template

```python
class Solution:
    def limitOccurrences(self, nums: List[int], k: int) -> List[int]:
        res = []
        for x in nums:
            if len(res) < k or res[-k] != x:   # chưa đủ k bản sao của x ở cuối res
                res.append(x)
        return res
```

Tại chỗ: `w = 0; for x in nums: if w < k or nums[w - k] != x: nums[w] = x; w += 1`, rồi `nums[:w]`.

### Checklist

- [ ] Đã đọc kỹ constraints (độ dài, miền giá trị) và biết thuật toán chạy được trong giới hạn đó.
- [ ] Đã chạy tay ít nhất **một ví dụ của đề** và một **trường hợp biên** (nhỏ nhất, lớn nhất, giá trị đặc biệt).
- [ ] Đúng **tên class / tên hàm / kiểu trả về** như skeleton LeetCode (đúng `bool`, `int`, `list`...).
- [ ] Không dùng `input()`, không `print` thừa trong code nộp.
- [ ] Không làm hỏng dữ liệu đầu vào nếu đề không cho phép (sao chép trước khi sửa).
- [ ] Đã so sánh với một cách brute force trên input ngẫu nhiên nhỏ.
- [ ] Mảng thực sự **sắp xếp** (mẹo `res[-k]` chỉ đúng khi đó); có `len(res) < k` đứng trước `res[-k]`.
- [ ] Đã thử `k = 1`, `k = len(nums)`, mảng toàn một giá trị, mảng một phần tử.